## 1. Set up environment anđ import library

In [45]:
%pip install huggingface-hub==0.34.4 datasets pandas pyarrow tqdm python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [46]:
import os
from dotenv import load_dotenv
from huggingface_hub import login, HfApi

### Login huggingface

In [47]:
load_dotenv()
hf_token = os.getenv("HUGFACE_TOKEN")

if hf_token:
    login(token=hf_token)
    print("Successfully logged in to Hugging Face Hub!")
else:
    print("HUGFACE_TOKEN not found in environment variables. Please set it to log in.")

Successfully logged in to Hugging Face Hub!


##  2. Deep-Dive Theory

### 2.1 Model Cards & Dataset Cards Best Practices
Model Cards and Dataset Cards are the soul of open-source documentation. They are not merely static README.md files, but also contain a YAML Frontmatter (metadata) section at the top, which is automatically parsed by the Hugging Face Hub to display interactive widgets, task tags, licenses, links, and metric table integrations.

#### 🔹 Standard YAML Metadata Structure for a Model Card:
```yaml
----
language:
- vi
- en
license: apache-2.0
tags:
- text-classification
- pytorch
datasets:
- dair-ai/emotion
metrics:
- accuracy
- f1
model-index:
- name: MyDebertaEmotionClassifier
  results:
  - task:
      type: text-classification
      name: Emotion Recognition
    dataset:
      type: dair-ai/emotion
      name: Emotion Dataset
      split: test
    metrics:
    - type: accuracy
      value: 0.9395
      name: Accuracy
    - type: f1
      value: 0.9169
      name: F1-Score
----
```

### 2.2 Version Control & Git Revisioning
Every repository on the Hugging Face Hub is essentially a **Git Repository** integrated with **Git LFS (Large File Storage)** to manage large weight files (`.bin`, `.safetensors`) without bloating the Git database size.

- **Commit Hashes:** Every time you push new model weights, the Hugging Face Hub creates a unique git commit hash. You can use this hash as a fixed `revision` to ensure that production systems are not accidentally affected by future updates.
- **Tags & Branches:** You can create tags such as `v1.0`, `v2.0`, or branches like `dev` and `main` directly on the Hugging Face Hub.
- **How to load a specific version:**
  ```python
  model = AutoModelForCausalLM.from_pretrained("user/model", revision="v1.0")
  ```

### 2.3 Private & Gated Repositories
- **Private Repos:** Only the repository owner or authorized members within an Organization can view or download the model. This is extremely useful for internal enterprise projects.
- **Gated Repos:** Allow models to remain publicly visible, but users must either click **Agree to the Terms and Conditions** or wait for manual approval from the administrator before gaining download access. Typical examples include models such as *Llama 3* and *Gemma*.

### 2.4 Hugging Face Organizations
Hugging Face allows users to create **Organizations**, similar to GitHub organizations. An Organization helps:
- Group multiple projects/models under a shared namespace (for example: `google`, `facebook`, `vietai`).
- Manage permissions with clearly defined roles: **Admin** (full access), **Write** (can push code/models), and **Read** (read-only access).
- Share computing resource quotas collectively (billing, Spaces hardware upgrades).

### 2.5 Dataset Streaming
When working with extremely large datasets (for example: hundreds of GBs or even TBs of text for LLM pretraining), downloading the entire dataset to disk or loading it fully into RAM becomes impractical on personal hardware.

- **How it works:** Instead of downloading compressed files, extracting them, and storing them locally, the `datasets` library uses **Streaming Mode** (`streaming=True`). Data is streamed directly from the Hugging Face Hub servers over HTTP as *Generators (yield)*, loading each sample into RAM only when requested by the training loop.
- **Advantages:** Training can start immediately without waiting hours for downloads, RAM usage remains extremely low, and almost no disk storage is required.

##  3. Hands-On Programming


In [48]:
api = HfApi()

user_info = api.whoami()
username = user_info["name"]
print(f"Đang thao tác dưới tài khoản: {username}")
print(f"Email đăng ký: {user_info.get('email')}")

Đang thao tác dưới tài khoản: HalogenFlo
Email đăng ký: None


### 3.2 Init Model Repository

In [49]:
model_repo_name = f"{username}/step8-model-demo"

try:
    repo_url = api.create_repo(
        repo_id=model_repo_name,
        repo_type="model",
        private=False,
        exist_ok=True
    )
    print(f"Repo '{model_repo_name}' has been created or already exists.")
    print(f"URL: {repo_url}")
except Exception as e:
    print(f"Error: {e}")

Repo 'HalogenFlo/step8-model-demo' has been created or already exists.
URL: https://huggingface.co/HalogenFlo/step8-model-demo


### 3.3 Push dummy model and Model Card



In [50]:
import json
import torch

local_model_dir = "./dummy_model"
os.makedirs(local_model_dir, exist_ok=True)

dummy_config = {
    "architectures": ["SimpleDenseClassifier"],
    "hidden_size": 256,
    "num_labels": 2,
    "model_type": "dense_classifier"
}
with open(os.path.join(local_model_dir, "config.json"), "w") as f:
    json.dump(dummy_config, f, indent=4)

dummy_weights = {
    "classifier.weight": torch.randn(2, 256),
    "classifier.bias": torch.randn(2)
}
torch.save(dummy_weights, os.path.join(local_model_dir, "pytorch_model.bin"))

model_card_content = """---
language:
- vi
- en
license: apache-2.0
tags:
- text-classification
- pytorch
- dummy-model
metrics:
- accuracy
model-index:
- name: Step8-Dummy-Classifier
  results:
  - task:
      type: text-classification
      name: Custom Text Classification
    dataset:
      type: custom
      name: Dummy Dataset
      split: test
    metrics:
    - type: accuracy
      value: 0.9500
      name: Accuracy
---

# Step 8 Dummy Classifier

```python
from transformers import AutoConfig
config = AutoConfig.from_pretrained("{model_repo_id}")
print(config)
```
""".format(model_repo_id=model_repo_name)

with open(os.path.join(local_model_dir, "README.md"), "w", encoding="utf-8") as f:
    f.write(model_card_content)
    
print(" Uploading folder dummy_model...")
future = api.upload_folder(
    folder_path=local_model_dir,
    repo_id=model_repo_name,
    repo_type="model",
    commit_message="Upload dummy model structure & model card"
)
print(f"Upload submitted. Future result: {future}")

 Uploading folder dummy_model...


pytorch_model.bin: 100%|██████████| 3.99k/3.99k [00:00<00:00, 9.05kB/s]


Upload submitted. Future result: https://huggingface.co/HalogenFlo/step8-model-demo/commit/ad5f752e9e7ba78b62cf234ce68178efba8d7a7d


### 3.4 Create and push Custom Dataset


In [51]:
import pandas as pd
from datasets import Dataset

raw_data = {
    "id": [1, 2, 3, 4, 5],
    "text": [
        "Mô hình này chạy rất nhanh và ổn định.",
        "Giao diện Hugging Face Spaces thiết kế cực đẹp!",
        "VRAM bị quá tải khi chạy full fine-tuning.",
        "Dataset streaming giúp tối ưu bộ nhớ RAM tuyệt đối.",
        "Hugging Face Hub là thư viện nguồn mở tuyệt vời cho AI Engineer."
    ],
    "label": [1, 1, 0, 1, 1]
}
df = pd.DataFrame(raw_data)

hf_dataset = Dataset.from_pandas(df)
print("Info of the created Dataset:")
print(hf_dataset)

dataset_repo_name = f"{username}/step8-dataset-demo"


print(f"Uploading dataset to: {dataset_repo_name}...")
hf_dataset.push_to_hub(
    repo_id=dataset_repo_name,
    commit_message="Initial upload of custom step8 dataset"
)
print("Successfully uploaded dataset to Hugging Face Hub!")

dataset_card_content = """---
language:
- vi
license: apache-2.0
size_categories:
- n<1K
task_categories:
- text-classification
pretty_name: Step 8 Custom Dataset Demo
---

# Step 8 Custom Dataset Demo

Sample dataset used for **Step 8: Model and Dataset Management**.

## Data Structure
Each data sample contains the following fields:
- `id`: Integer identifier
- `text`: Sample Vietnamese text
- `label`: Binary sentiment label (1: Positive, 0: Negative)

## How to Use
```python
from datasets import load_dataset
dataset = load_dataset("{dataset_repo_id}")
print(dataset['train'][0])
```
""".format(dataset_repo_id=dataset_repo_name)

api.upload_file(
    path_or_fileobj=dataset_card_content.encode("utf-8"),
    path_in_repo="README.md",
    repo_id=dataset_repo_name,
    repo_type="dataset",
    commit_message="Add comprehensive Dataset Card"
)
print("Successfully updated Dataset Card!")

Info of the created Dataset:
Dataset({
    features: ['id', 'text', 'label'],
    num_rows: 5
})
Uploading dataset to: HalogenFlo/step8-dataset-demo...


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 498.08ba/s]
tmp70wepoa0.parquet: 100%|██████████| 2.19k/2.19k [00:00<00:00, 7.00kB/s]
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.34s/ shards]


Successfully uploaded dataset to Hugging Face Hub!
Successfully updated Dataset Card!


### 3.5 Dataset Streaming


In [52]:
import time
import psutil
from datasets import load_dataset

def get_ram_usage():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

dataset_name = "stanfordnlp/imdb"

print("---  1: STANDARD LOAD  ---")
ram_before = get_ram_usage()
start_time = time.time()

standard_ds = load_dataset(dataset_name, split="train")

duration_standard = time.time() - start_time
ram_after = get_ram_usage()
ram_used_standard = ram_after - ram_before

print(f"Time loading: {duration_standard:.2f} s")
print(f"RAM: {ram_used_standard:.2f} MB")
print(f"Number of samples: {len(standard_ds)}")
print(f"First sample: {standard_ds[0]['text'][:100]}...")

print("\n" + "="*50 + "\n")

print("---  2: DATASET STREAMING ---")
ram_before_stream = get_ram_usage()
start_time_stream = time.time()

stream_ds = load_dataset(dataset_name, split="train", streaming=True)

duration_stream = time.time() - start_time_stream
ram_after_stream = get_ram_usage()
ram_used_stream = ram_after_stream - ram_before_stream

print(f"Time loading: {duration_stream:.4f} s")
print(f"RAM: {ram_used_stream:.2f} MB ")

first_sample = next(iter(stream_ds))
print(f"Example: {first_sample['text'][:100]}...")

---  1: STANDARD LOAD  ---
Time loading: 2.25 s
RAM: 0.11 MB
Number of samples: 25000
First sample: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it w...


---  2: DATASET STREAMING ---
Time loading: 1.6824 s
RAM: 0.00 MB 
Example: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it w...


### 3.6  Git Revision / Branch 

In [53]:
from transformers import AutoConfig

try:
    api.create_tag(
        repo_id=model_repo_name,
        repo_type="model",
        tag="v1.0",
        commit_message="Release version 1.0 of dummy classifier"
    )
    print(f"Successfully created tag 'v1.0' on repository {model_repo_name}!")
    
    config_from_tag = AutoConfig.from_pretrained(
        model_repo_name,
        revision="v1.0"       
    )
    print("Successfully loaded config from Tag v1.0!")
    print(f"   - Type: {config_from_tag.model_type}")
    print(f"   - Hidden size: {config_from_tag.hidden_size}")
    
except Exception as e:
    print(f"Error occurred: {e}")

Error occurred: HfApi.create_tag() got an unexpected keyword argument 'commit_message'


##  4. Cleanup


In [54]:
api.delete_repo(repo_id=model_repo_name, repo_type="model")
print(f"Repository deleted: {model_repo_name}")

api.delete_repo(repo_id=dataset_repo_name, repo_type="dataset")
print(f"Repository deleted: {dataset_repo_name}")
print("Successfully cleaned up all created repositories.")

Repository deleted: HalogenFlo/step8-model-demo
Repository deleted: HalogenFlo/step8-dataset-demo
Successfully cleaned up all created repositories.
